In [ ]:
# config.py
from pathlib import Path

# Directory where SHAP outputs and artifacts are stored.
RESULTS_DIR = Path("SHAP")

# Patterns for summary CSV and metrics JSON filenames.
CSV_SUMMARY_PATTERN = "iter_{:04}_summary.csv"
JSON_METRICS_PATTERN = "iter_{:04}_metrics.json"
FEATURE_IMPORTANCE_CSV = "feature_importance_values.csv"

# Output settings
REPORT_NAME = "final_analysis_report.md"
OUTPUT_IMAGE_NAME = "feature_importance_evolution.png"

# Plotting palette
COLOR_PALETTE = [
    "#3A7A94", "#C85A5A", "#1E5631", "#0C3965", "#A8D8A8",
    "#D5A64A", "#8B5A8C", "#5A8B8B", "#A87A5A", "#5A7AD5"
]


In [ ]:
from pathlib import Path
import pandas as pd
import json
from typing import List, Tuple, Dict, Any


def iter_dirs(root: Path) -> List[Path]:
    """Scan root for iteration directories named 'iter_<number>'.

    Args:
        root: Directory containing iteration subdirectories.

    Returns:
        Sorted list of Path objects for each iteration directory.
    """
    dirs = [d for d in root.glob("iter_*") if d.is_dir()]
    return sorted(dirs, key=lambda p: int(p.name.split("_")[1]))

def load_feature_importance(results_dir: Path) -> pd.DataFrame:
    """Load and aggregate feature importance CSV files from iteration subdirectories.

    Args:
        results_dir: Root directory containing iteration subfolders.

    Returns:
        DataFrame with aggregated feature importance and iteration column.
    """
    records: List[pd.DataFrame] = []
    for folder in iter_dirs(results_dir):
        iteration = int(folder.name.split("_")[1])
        file_path = folder / FEATURE_IMPORTANCE_CSV
        if file_path.exists():
            df = pd.read_csv(file_path)
            df["iteration"] = iteration
            records.append(df)
    if not records:
        raise FileNotFoundError(f"No feature importance files found in {results_dir}")
    return pd.concat(records, ignore_index=True)

def load_iteration_summary(folder: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Load summary CSV and metrics JSON for a single iteration.

    Args:
        folder: Path to an iteration directory.

    Returns:
        Tuple of (DataFrame, metrics dict) for that iteration.
    """
    iteration = int(folder.name.split("_")[1])
    csv_path = folder / CSV_SUMMARY_PATTERN.format(iteration)
    json_path = folder / JSON_METRICS_PATTERN.format(iteration)
    if not csv_path.exists() or not json_path.exists():
        raise FileNotFoundError(f"Missing summary or metrics in {folder}")
    df = pd.read_csv(csv_path).assign(iteration=iteration)
    metrics = json.loads(json_path.read_text())
    metrics["iteration"] = iteration
    return df, metrics

def load_all_iterations(root: Path) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    """Load summaries and metrics from all iteration directories.

    Args:
        root: Directory containing iteration subfolders.

    Returns:
        Combined DataFrame of all iteration summaries and list of metrics dicts.
    """
    dfs: List[pd.DataFrame] = []
    metrics_list: List[Dict[str, Any]] = []
    for folder in iter_dirs(root):
        df, metrics = load_iteration_summary(folder)
        dfs.append(df)
        metrics_list.append(metrics)
    if not dfs:
        raise FileNotFoundError(f"No iteration directories found in {root}")
    full_df = pd.concat(dfs, ignore_index=True)
    return full_df, metrics_list

In [ ]:
# plotter.py
from pathlib import Path
from typing import List, Optional
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

def pivot_and_normalize(df: pd.DataFrame) -> pd.DataFrame:
    """Pivot DataFrame to have features as columns and normalize by iteration.

    Args:
        df: DataFrame containing 'feature', 'mean_abs_shap', and 'iteration'.

    Returns:
        Normalized DataFrame with iterations as index and features as columns.
    """
    pivot = df.pivot(index="iteration", columns="feature", values="mean_abs_shap")
    return pivot.div(pivot.sum(axis=1), axis=0)

def plot_evolution(
    df: pd.DataFrame,
    well_name: str,
    palette: Optional[List[str]] = None,
    save_dir: Optional[Path] = None,
    save_name: Optional[str] = None
) -> None:
    """Plot and save the evolution of SHAP feature importance.

    Args:
        df: Aggregated DataFrame with feature importances.
        well_name: Identifier for the plot title context.
        palette: List of hex color codes for plot lines.
        save_dir: Directory to save the static image.
        save_name: Filename for the saved image.
    """
    palette = palette or COLOR_PALETTE
    save_dir = save_dir or RESULTS_DIR
    save_name = save_name or OUTPUT_IMAGE_NAME

    normalized = pivot_and_normalize(df)
    avg_contrib = normalized.mean().sort_values(ascending=False)
    features = avg_contrib.index.tolist()

    fig = go.Figure()
    for idx, feature in enumerate(features):
        fig.add_trace(go.Scatter(
            x=normalized.index,
            y=normalized[feature],
            mode="lines+markers",
            name=f"{feature} ({avg_contrib[feature]:.2%})",
            line=dict(color=palette[idx % len(palette)], width=3),
            marker=dict(size=8),
            hovertemplate=(
                f"<b>Feature:</b> {feature}<br>"
                "<b>Iteration:</b> %{x}<br>"
                "<b>Importance:</b> %{y:.2%}<extra></extra>"
            )
        ))

    fig.update_layout(
        title=dict(
            text=f"<b>Evolution of Relative Feature Importance (SHAP)</b><br>Well: {well_name}",
            x=0.5, xanchor="center", font=dict(size=36, color="#3A7A94")
        ),
        xaxis=dict(
            title="Training Iteration",
            tickfont_size=24,
            gridcolor="rgba(230, 230, 230, 0.5)",
            zeroline=False,
            title_font=dict(size=28, color="#0C3965"),
            domain=[0, 0.95]
        ),
        yaxis=dict(
            title="Normalized Mean |SHAP| Value",
            tickfont_size=24,
            gridcolor="rgba(230, 230, 230, 0.5)",
            tickformat=".0%",
            zeroline=False,
            title_font=dict(size=28, color="#0C3965")
        ),
        legend=dict(
            font=dict(size=24),
            orientation="h",
            yanchor="bottom", y=0.1,
            xanchor="left", x=1.01
        ),
        font=dict(family="Arial, sans-serif"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        width=1200, height=800,
        hovermode="x unified",
        margin=dict(l=150, r=150, t=150, b=150)
    )

    fig.show()

    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / save_name
    pio.write_image(fig, str(save_path), format="png", width=1200, height=800, scale=3)
    print(f"High-resolution plot saved to: {save_path}")


In [ ]:
import pandas as pd
import textwrap
from typing import List, Dict, Any

def gini_narr(metrics: List[Dict[str, Any]]) -> str:
    """Generate narrative on Gini coefficient trend."""
    initial = metrics[0]["gini"]
    final = metrics[-1]["gini"]
    if final > initial * 1.1:
        trend = "increasing → model relies on fewer key features"
    elif final < initial * 0.9:
        trend = "decreasing → importance more evenly spread"
    else:
        trend = "stable"
    return f"Gini went from {initial:.2f} to {final:.2f} ({trend})."

def convergence_narr(metrics: List[Dict[str, Any]]) -> str:
    """Generate narrative on feature ranking convergence."""
    for entry in metrics[1:]:
        rho = entry.get("spearman_vs_prev")
        if rho is not None and rho > 0.9:
            return (
                f"Feature ranking stabilized around iteration {entry['iteration']} "
                f"(Spearman ≈ {rho:.2f})."
            )
    return "Ranking never stabilized (Spearman < 0.9)."

def top_features_narr(full_df: pd.DataFrame, metrics: List[Dict[str, Any]]) -> str:
    """Generate narrative on top influencing features."""
    avg_rank = full_df.groupby("feature")["rank"].mean().sort_values()
    top3 = avg_rank.head(3).index.tolist()
    count_top1 = sum(m.get("top_feature") == top3[0] for m in metrics)
    top_list = ", ".join(top3)
    return f"Most influential: **{top_list}**. '{top3[0]}' ranked #1 in {count_top1} iterations."

def key_drivers_narr(full_df: pd.DataFrame) -> Dict[str, Any]:
    """Identify top positive and negative feature drivers."""
    avg_importance = full_df.groupby("feature")["mean_|SHAP|"].mean()
    directions = full_df.groupby("feature")["direction"].first()
    drivers_df = pd.concat([avg_importance, directions], axis=1).reset_index()
    drivers_df.columns = ["feature", "mean_abs_shap", "direction"]
    drivers_df.sort_values(by="mean_abs_shap", ascending=False, inplace=True)
    positive = drivers_df[drivers_df["direction"] == "↑"]["feature"].head(2).tolist()
    negative = drivers_df[drivers_df["direction"] == "↓"]["feature"].head(2).tolist()
    return {"positive": positive, "negative": negative}

def build_insights(
    full_df: pd.DataFrame,
    metrics: List[Dict[str, Any]]
) -> Dict[str, Any]:
    """Build a dictionary of narrative insights from data and metrics."""
    sorted_metrics = sorted(metrics, key=lambda x: x["iteration"])
    return {
        "gini": gini_narr(sorted_metrics),
        "convergence": convergence_narr(sorted_metrics),
        "top": top_features_narr(full_df, sorted_metrics),
        "drivers": key_drivers_narr(full_df)
    }

def render_report(insights: Dict[str, Any], metrics: List[Dict[str, Any]]) -> str:
    """Render a Markdown report from insights and iteration metrics."""
    metrics_table = pd.DataFrame(metrics).set_index("iteration").to_markdown()
    pos = insights["drivers"]["positive"]
    neg = insights["drivers"]["negative"]
    pos_str = " and ".join(f"**{f}**" for f in pos[:2])
    neg_str = " and ".join(f"**{f}**" for f in neg[:2])
    report = textwrap.dedent(f"""
        # Automated SHAP Analysis Report

        **Top Influencers** — {insights['top']}

        **Model Focus** — {insights['gini']}

        **Learning Stability** — {insights['convergence']}

        **Key Drivers Analysis**:
        * Positive drivers: {pos_str}.
        * Negative drivers: {neg_str}.

        ---
        ## Metrics by Iteration
        {metrics_table}

        ---
        *Generated automatically.*
    """).strip()
    return report


In [ ]:
# main.py
import logging

def main() -> None:
    """Execute SHAP analysis plotting and report generation."""
    shap_df = load_feature_importance(RESULTS_DIR)
    plot_evolution(shap_df, well_name="15/9-F-12")
    full_df, metrics = load_all_iterations(RESULTS_DIR)
    insights = build_insights(full_df, metrics)
    report = render_report(insights, metrics)
    report_path = RESULTS_DIR / REPORT_NAME
    report_path.write_text(report)
    logging.info(f"Report saved to {report_path}")
if __name__ == "__main__":
    main()
